In [1]:
import sys
import os
import pandas as pd

# 1. Path fix to load your custom src modules
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
from src.geocoder_ensemble import reposition_with_ensemble
from src.metrics import haversine_meters, improvement_summary

# 2. Load API keys for Google Maps and Mapbox
load_dotenv(os.path.abspath('../.env'))

# 3. Load the valid ground truth data
gt_df = pd.read_parquet("../data/processed/ground_truth.parquet")
valid_gt = gt_df.dropna(subset=["gt_lat", "gt_lon"]).copy()

print(f"Running Ensemble on {len(valid_gt)} places... (This will take a moment)")

# 4. Run the multi-geocoder ensemble
# Using the 25m agreement radius defined in your project plan
results_df = reposition_with_ensemble(valid_gt, agreement_radius_m=25.0)

# 5. Calculate the new spatial offset between the Ensemble and the LLM Ground Truth
results_df["ensemble_offset_m"] = results_df.apply(
    lambda r: haversine_meters(r["ensemble_lat"], r["ensemble_lon"], r["gt_lat"], r["gt_lon"])
    if pd.notnull(r["ensemble_lat"]) else None,
    axis=1
)

# Filter out any places where the geocoders completely failed
valid_results = results_df.dropna(subset=["ensemble_offset_m"]).copy()
print(f"Successfully repositioned {len(valid_results)} places.")

# 6. Measure the Improvement vs Baseline
summary = improvement_summary(
    baseline_offsets=valid_results["offset_haversine_m"],
    new_offsets=valid_results["ensemble_offset_m"]
)

print("\n=== Method 1: Multi-Geocoder Ensemble Results ===")
print(f"Baseline Median: {summary['baseline']['median_m']}m")
print(f"Ensemble Median: {summary['new_method']['median_m']}m")
print(f"Improvement:     {summary['median_improvement_m']}m ({summary['median_improvement_pct']}%)")
print(f"Regression Rate: {summary['regression_rate_pct']}% (Pins made worse)")

Running Ensemble on 750 places... (This will take a moment)


US Census geocoding failed for 'Chicago O'Hare International Airport, Terminal 3, Gate K15 - Food Court, 10000 Bessie Coleman Dr, 60666, US': 400 Client Error:  for url: https://geocoding.geo.census.gov/geocoder/locations/onelineaddress?address=Chicago+O%27Hare+International+Airport%2C+Terminal+3%2C+Gate+K15+-+Food+Court%2C+10000+Bessie+Coleman+Dr%2C+60666%2C+US&benchmark=Public_AR_Current&format=json
US Census geocoding failed for '8005 Harford Rd, Parkville, MD, 21234-5753, US': HTTPSConnectionPool(host='geocoding.geo.census.gov', port=443): Read timed out. (read timeout=10)
US Census geocoding failed for '600 Mule Rd, Toms River, NJ, 08757-6460, US': HTTPSConnectionPool(host='geocoding.geo.census.gov', port=443): Read timed out. (read timeout=10)


Successfully repositioned 724 places.

=== Method 1: Multi-Geocoder Ensemble Results ===
Baseline Median: 0.0m
Ensemble Median: 68.24m
Improvement:     -68.24m (0%)
Regression Rate: 98.34% (Pins made worse)


In [5]:
from src.features import build_training_data
import xgboost as xgb
from sklearn.model_selection import train_test_split

print("Generating ML candidates and extracting features...")

# Create a dictionary mapping place IDs to their geocoder results
geocoder_map = {}
for _, row in valid_results.iterrows():
    if pd.notnull(row["ensemble_lat"]):
        geocoder_map[row["id"]] = [{"lat": row["ensemble_lat"], "lon": row["ensemble_lon"], "source": row["ensemble_method"]}]

# FIX: Rename the columns so features.py can find the original pin coordinates
valid_gt_features = valid_gt.rename(columns={
    "current_lat": "lat", 
    "current_lon": "lon"
})

# Build the feature matrix and labels using the renamed dataframe
features_df, labels = build_training_data(
    gt_df=valid_gt_features,
    geocode_results_map=geocoder_map,
    building_centroids_map=None # We can add OSM building footprints later if needed
)

# Prepare for XGBoost
X = features_df.drop(columns=["place_id", "candidate_lat", "candidate_lon", "candidate_source"])
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining data ready: {X_train.shape[0]} candidate rows.")
print(f"Features extracted: {list(X.columns)}")

Generating ML candidates and extracting features...

Training data ready: 1179 candidate rows.
Features extracted: ['dist_from_pin_m', 'avg_geocode_dist_m', 'min_building_dist_m', 'n_buildings_nearby', 'n_geocoders', 'confidence', 'source_count', 'is_current_pin', 'is_geocode', 'is_building', 'cat_food', 'cat_lodging', 'cat_retail', 'cat_services', 'cat_health', 'cat_auto']


In [6]:
print("Training XGBoost Candidate Ranker...")
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
print("Model trained! Evaluating performance...")

features_df['pred_prob'] = model.predict_proba(X)[:, 1]

best_candidates = features_df.loc[features_df.groupby('place_id')['pred_prob'].idxmax()]

eval_df = pd.merge(
    valid_gt_features,
    best_candidates[['place_id', 'candidate_lat', 'candidate_lon', 'candidate_source']],
    left_on='id',
    right_on='place_id',
    how='inner'
)

eval_df["ml_offset_m"] = eval_df.apply(
    lambda r: haversine_meters(r["candidate_lat"], r["candidate_lon"], r["gt_lat"], r["gt_lon"]),
    axis=1
)

summary_ml = improvement_summary(
    baseline_offsets=eval_df["offset_haversine_m"],
    new_offsets=eval_df["ml_offset_m"]
)

print("\n=== Method 2: ML Candidate Ranking Results ===")
print(f"Baseline Median: {summary_ml['baseline']['median_m']}m")
print(f"ML Method Median: {summary_ml['new_method']['median_m']}m")
print(f"Improvement:     {summary_ml['median_improvement_m']}m ({summary_ml['median_improvement_pct']}%)")
print(f"Regression Rate: {summary_ml['regression_rate_pct']}% (Pins made worse)")

print("\n=== Where did the ML model move the pins? ===")
print(eval_df['candidate_source'].value_counts())

Training XGBoost Candidate Ranker...
Model trained! Evaluating performance...

=== Method 2: ML Candidate Ranking Results ===
Baseline Median: 0.0m
ML Method Median: 0.0m
Improvement:     0.0m (0%)
Regression Rate: 0.27% (Pins made worse)

=== Where did the ML model move the pins? ===
candidate_source
current_pin                 746
geocode_weighted_average      3
geocode_median                1
Name: count, dtype: int64
